# Martingale Trading, Backtested on Crypto
Martingale doubles the bet after every loss, so the first win recovers everything. Trading bots
sell the same idea as "DCA (dollar-cost averaging) with a volume scale of 2": buy, and every time the price falls a step,
buy twice as much; sell the whole stack a little above its average price. We build it on Bitcoin
perpetual futures from 2020 to 2026 — without leverage, then with it, where an exchange liquidates
the stack. Education, not financial advice.

## 1. Install
freqtrade is the backtester; we call only a few of its methods.

In [ ]:
!pip install -q freqtrade==2026.10

## 2. Helpers from the earlier episodes
`binance_vision` — raw Binance data with SHA-256 hash checks (episode q00); `quant_report` — the verdict and hourly marked-to-market equity (episode q01).

In [ ]:
!curl -fsSLO https://raw.githubusercontent.com/nikshev/unstuck/main/episodes/quant-lib/binance_vision.py -O https://raw.githubusercontent.com/nikshev/unstuck/main/episodes/quant-lib/quant_report.py
from binance_vision import save_for_freqtrade, DATA
from quant_report import verdict, equity, drawdown
import json
import pandas as pd
import matplotlib.pyplot as plt

## 3. The data: Bitcoin, hourly, 2020 to 2026
Hourly candles, because a bot reacts within the day; from January 2020, so the crash of 12 March 2020 is in the test. Uses `save_for_freqtrade`, `DATA` (section 2).

In [ ]:
save_for_freqtrade("BTCUSDT", "1h", "2020-01", "2026-10")
btc = pd.read_feather(DATA / "BTC_USDT_USDT-1h-futures.feather")
btc = btc.set_index("date")
print(btc["close"].iloc[[0, -1]].to_string())
print("hours:", len(btc))

## 4. The ladder
$n$ buys, each twice the last: total $B(1 + 2 + \dots + 2^{n-1}) = B(2^n - 1)$. To make the whole stack fit the account, $B = \dfrac{\text{account}}{2^n - 1}$. Each buy 3% below the previous one; the exit is 1% above the average price.

In [ ]:
account, n, step, take = 10_000, 7, 0.03, 0.01
B = account / (2 ** n - 1)                         # the first buy
ladder = pd.DataFrame({"buy": range(1, n + 1)})
ladder["price"] = (1 - step) ** (ladder["buy"] - 1)    # 1.00 = first price
ladder["stake"] = B * 2 ** (ladder["buy"] - 1)
ladder["total"] = ladder["stake"].cumsum()
coins = (ladder["stake"] / ladder["price"]).cumsum()
ladder["average"] = ladder["total"] / coins            # average price paid
ladder["exit at"] = ladder["average"] * (1 + take)
print(f"B = {B:.2f} USDT")
print(ladder.round(3).to_string(index=False))

## 5. Backtest settings
Isolated margin: the stack's margin is all it can lose. One position at a time, 10,000 USDT (Tether, a dollar stablecoin), Binance's fee 0.05%.

In [ ]:
config = {
    "trading_mode": "futures",
    "margin_mode": "isolated",                # the stack's own margin
    "liquidation_buffer": 0.0,
    "stake_currency": "USDT",
    "stake_amount": "unlimited",              # the strategy sizes every buy
    "dry_run_wallet": 10000, "dry_run": True,
    "max_open_trades": 1,                     # one stack at a time
    "timeframe": "1h", "fee": 0.0005,
    "entry_pricing": {"price_side": "same", "use_order_book": True,
                      "order_book_top": 1},
    "exit_pricing": {"price_side": "same", "use_order_book": True,
                     "order_book_top": 1},
    "exchange": {"name": "binance",
                 "pair_whitelist": ["BTC/USDT:USDT"]},
    "pairlists": [{"method": "StaticPairList"}],
}
json.dump(config, open("config.json", "w"), indent=1)

## 6. Strategy file, part 1: the settings
The ladder from section 4. `MMR` = maintenance margin rate, Binance's 0.4% for Bitcoin at this size (episode q00).

In [ ]:
%%writefile user_data/strategies/Martingale.py
from freqtrade.strategy import IStrategy, stoploss_from_absolute

MMR = 0.004                                    # maintenance margin rate

class Martingale(IStrategy):
    ACCOUNT, N, STEP, TAKE, LEV = 10_000, 7, 0.03, 0.01, 1   # section 4
    timeframe = "1h"
    minimal_roi = {"0": 100}                   # the exit is ours, below
    stoploss = -0.99                           # martingale never stops out
    position_adjustment_enable = True          # needed to buy more
    max_entry_position_adjustment = 6          # 1 + 6 = 7 buys
    use_custom_stoploss = True                 # for the liquidation price

## 7. Part 2: always in the market
A new stack opens on the first candle after the last one closed; its first buy is B. `ACCOUNT`, `N`, `LEV` from section 6.

In [ ]:
%%writefile -a user_data/strategies/Martingale.py

    def populate_indicators(self, df, metadata):
        return df

    def populate_entry_trend(self, df, metadata):
        df["enter_long"] = 1                   # always ready to start again
        return df

    def populate_exit_trend(self, df, metadata):
        return df

    def leverage(self, pair, **kwargs):
        return self.LEV

    def custom_stake_amount(self, pair, current_time, current_rate,
                            proposed_stake, min_stake, max_stake, leverage,
                            entry_tag, side, **kwargs):
        return self.ACCOUNT / (2 ** self.N - 1)   # B, the first buy

## 8. Part 3: double down
When the price is `STEP` below the last buy, buy $B \cdot 2^k$, k = buys so far — twice the last one. freqtrade calls this every candle while the stack is open. Settings from section 6.

In [ ]:
%%writefile -a user_data/strategies/Martingale.py

    def adjust_trade_position(self, trade, current_time, current_rate,
                              current_profit, min_stake, max_stake,
                              current_entry_rate, current_exit_rate,
                              current_entry_profit, current_exit_profit,
                              **kwargs):
        if trade.has_open_orders:
            return None
        buys = trade.select_filled_orders(trade.entry_side)
        last = buys[-1]                              # the most recent buy
        if current_rate > last.safe_price * (1 - self.STEP):
            return None                              # not a step lower yet
        # B * 2^k from the plan, not 2 x the last fill: the exchange rounds
        # every order to 0.001 BTC, and doubling a rounded fill compounds it
        stake = self.ACCOUNT / (2 ** self.N - 1) * 2 ** len(buys)
        return min(stake, max_stake)

## 9. Part 4: the exit, and the liquidation price
Sell the stack 1% above its average price, `trade.open_rate`. Isolated long: $P_{liq} = E \cdot \dfrac{1 - m/L}{1 - MMR}$, E = average entry price, L = leverage, m = margin left after funding, MMR = maintenance margin rate (episode q00). Right after a buy, freqtrade applies the stop's percentage to the new average price, so then we give it relative to that price. `MMR` and `TAKE` from section 6.

In [ ]:
%%writefile -a user_data/strategies/Martingale.py

    def custom_exit(self, pair, trade, current_time, current_rate,
                    current_profit, **kwargs):
        if current_rate >= trade.open_rate * (1 + self.TAKE):
            return "take_profit"                 # 1% above the average

    def custom_stoploss(self, pair, trade, current_time, current_rate,
                        current_profit, after_fill, **kwargs):
        m = 1 + (trade.funding_fees or 0) / trade.stake_amount
        liq = trade.open_rate * (1 - m / trade.leverage) / (1 - MMR)
        if liq <= 0:
            return None                          # 1x: no liquidation price
        # after a buy freqtrade re-applies this % to the new average price
        ref = trade.open_rate if after_fill else current_rate
        return stoploss_from_absolute(liq, ref, is_short=False,
                                      leverage=trade.leverage)

## 10. Run it, no leverage
From 1 January 2020 to 31 October 2026. Uses `config.json` (5), `Martingale` (6–9), `verdict`, `equity`, `drawdown` (2).

In [ ]:
!freqtrade backtesting --cache none -c config.json -s Martingale --timerange 20200101-20261031 --export trades > run.log 2>&1
plain, trades = verdict("Martingale")
plain["drawdown, marked to market"] = drawdown(equity(trades))

## 11. What the stacks looked like
How many buys each stack needed, how long it stayed open, and the longest wait — buy by buy. Uses `trades` (section 10).

In [ ]:
trades["buys"] = trades["orders"].map(
    lambda o: sum(x["ft_is_entry"] for x in o))
trades["days"] = trades["trade_duration"] / 1440        # minutes -> days
print(trades["buys"].value_counts().sort_index().to_string())
print("won:", (trades["profit_abs"] > 0).sum(), "of", len(trades))
longest = trades.sort_values("days").iloc[-1]
print("longest stack:", longest["open_date"].date(), "->",
      longest["close_date"].date(), f"{longest['days']:.0f} days,",
      f"{longest['buys']} buys")
for o in longest["orders"]:
    if o["ft_is_entry"]:                     # the buys, one by one
        when = pd.to_datetime(o["order_filled_timestamp"], unit="ms")
        print(f"  buy {when.date()} at {o['safe_price']:>9,.1f}"
              f"  {o['cost']:>7,.0f} USDT")
print(f"stack {longest['stake_amount']:,.0f} USDT, average price "
      f"{longest['open_rate']:,.1f}, sold at {longest['close_rate']:,.1f}")
print(f"its profit {longest['profit_abs']:.0f} USDT, of which funding "
      f"{longest['funding_fees']:.0f}")

## 12. The same bot with leverage
3x and 5x: the same margin controls a bigger position, so every win is bigger — and the stack has a liquidation price. Subclasses of `Martingale` (section 6).

In [ ]:
%%writefile user_data/strategies/MartingaleLev.py
from Martingale import Martingale

class Martingale3x(Martingale):
    LEV = 3

class Martingale5x(Martingale):
    LEV = 5

### Run both
Uses `config.json` (5), the subclasses (above), `verdict`, `equity`, `drawdown` (2).

In [ ]:
runs, lev_trades = {}, {}
for name in ["Martingale3x", "Martingale5x"]:
    !freqtrade backtesting --cache none -c config.json -s {name} --timerange 20200101-20261031 --export trades > run.log 2>&1
    runs[name], lev_trades[name] = verdict(name)
    runs[name]["drawdown, marked to market"] = drawdown(
        equity(lev_trades[name]))
    liq = lev_trades[name][lev_trades[name]["exit_reason"]
                           .str.contains("stop|liquid")]
    print(name, "liquidations:", len(liq))
    print(liq[["open_date", "close_date", "profit_abs"]].to_string())
    worst = liq.loc[liq["profit_abs"].idxmin()]      # the biggest one
    print(f"  biggest: {worst['stake_amount']:,.0f} USDT,"
          f" average {worst['open_rate']:,.1f},"
          f" liquidated {worst['close_rate']:,.1f},"
          f" funding {worst['funding_fees']:,.0f}")

## 13. Three equity curves and Bitcoin
Every account valued hourly, against holding Bitcoin with the same 10,000. Uses `btc` (3), `trades` (10), `lev_trades` (12), `equity`, `drawdown` (2).

In [ ]:
hold = 10_000 * btc["close"] / btc["close"].iloc[0]
curves = {"hold Bitcoin": hold, "martingale 1x": equity(trades),
          "martingale 3x": equity(lev_trades["Martingale3x"]),
          "martingale 5x": equity(lev_trades["Martingale5x"])}
for name, curve in curves.items():
    print(f"{name:>14}: final {curve.iloc[-1]:>8.0f}"
          f"  lowest {curve.min():>7.0f}  max drawdown {drawdown(curve):.0%}")
    fall = 1 - curve / curve.cummax()          # below the peak so far
    bottom = fall.idxmax()
    top = curve.loc[:bottom].idxmax()
    print(f"{'':>16}peak {top.date()} {curve[top]:>7,.0f}"
          f"  bottom {bottom.date()} {curve[bottom]:>7,.0f}")
plt.figure(figsize=(11, 5))
for name, curve in curves.items():
    plt.plot(curve.index, curve.clip(lower=1), label=name)
plt.yscale("log"); plt.legend(); plt.title("10,000 USDT, 2020 to 2026")
plt.show()

## 14. The Sharpe ratio: return per unit of wobble
The same four lines as episode q03: Sharpe ratio $S = \dfrac{\text{mean of the daily returns}}{\text{their standard deviation}} \times \sqrt{365}$, the hourly account (section 13) taken once a day; risk-free rate 0: idle USDT (Tether, a dollar stablecoin) earns nothing here. Uses `curves` (13), `plain` (10), `runs` (12).

In [ ]:
def sharpe(curve):
    day = curve.resample("1D").last()           # the account once a day
    r = day.pct_change().dropna()               # daily returns
    return r.mean() / r.std() * 365 ** 0.5      # a year = 365 crypto days

for name, curve in curves.items():
    r = curve.resample("1D").last().pct_change().dropna()
    print(f"{name:>14}: mean {r.mean():.3%}  std {r.std():.2%}"
          f"  Sharpe {sharpe(curve):.2f}")
plain["Sharpe"] = sharpe(curves["martingale 1x"])
runs["Martingale3x"]["Sharpe"] = sharpe(curves["martingale 3x"])
runs["Martingale5x"]["Sharpe"] = sharpe(curves["martingale 5x"])

## 15. Every run side by side
A legend explains every row. Uses `plain` (10, 14), `runs` (12, 14).

In [ ]:
table = pd.DataFrame({"1x": plain, "3x": runs["Martingale3x"],
                      "5x": runs["Martingale5x"]})
table.index = ["start", "final", "CAGR", "drawdown (freqtrade)", "trades",
               "long profit", "short profit", "funding", "fees",
               "drawdown (hourly)", "Sharpe"]
pct = ["CAGR", "drawdown (freqtrade)", "drawdown (hourly)"]
def cell(row, v):                       # how each row is written
    if row in pct:
        return f"{v:.0%}"
    return f"{v:.2f}" if row == "Sharpe" else f"{v:,.0f}"
shown_table = table.apply(lambda r: r.map(lambda v: cell(r.name, v)), axis=1)
print(shown_table.drop(index="short profit").to_string())
print("money in USDT (Tether, a dollar stablecoin); 1x, 3x, 5x = leverage")
print("CAGR = compound annual growth rate: growth per year")
print("drawdown = deepest fall from a peak; freqtrade: closed trades only,")
print("  hourly: the whole account valued every hour")
print("funding = paid between longs and shorts every 8 hours")
print("fees = the exchange fee, 0.05% of every order")
print("Sharpe = mean daily return / its standard deviation x sqrt(365);")
print("  above 1 is the usual target")